In [15]:
from dotenv import load_dotenv
load_dotenv()

import kagglehub
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn, optim
from torch.optim import AdamW
import torch
from tqdm import tqdm
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, get_scheduler
from huggingface_hub import login
from datasets import Dataset

EMBED_DIM = 128
HIDDEN_DIM = 64
MAX_EPOCHS = 4

login(token=os.getenv("HF_TOKEN"))

path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
df = pd.read_csv(os.path.join(path, "Reviews.csv"), usecols=["Text", "Score"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [16]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_batch(batch):
    return tokenizer(batch['Text'], padding="max_length", truncation=True, max_length=128)

In [17]:
df["Text"] = df["Text"].str.replace(r"<[^>]+>", " ", regex=True)
df["Text"] = df["Text"].str.replace(r"[^\w\s]", " ", regex=True)
df["labels"] = df["Score"] - 1

In [18]:
# Panda and Hugging Face stuff 
ds = Dataset.from_pandas(df[["Text", "labels"]])
ds = ds.map(tokenize_batch, batched=True)
ds = ds.remove_columns(["Text"])
ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map: 100%|██████████| 568454/568454 [01:09<00:00, 8195.78 examples/s]


In [19]:
ds = ds.train_test_split(test_size=0.2, seed=42)
ds_train = ds["train"]
ds_val = ds["test"]

dataloader_train = DataLoader(ds_train, batch_size=32, shuffle=True)
dataloader_val = DataLoader(ds_val, batch_size=32)

In [20]:
device = torch.device("cpu")
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  device = torch.device("mps")

print(device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")

model = DistilBertForSequenceClassification.from_pretrained("prajjwal1/bert-mini", num_labels=5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=MAX_EPOCHS*len(dataloader_train))

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

cuda
NVIDIA GeForce RTX 4060 Ti


d:\LLM\trust-behaviours\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maxim\.cache\huggingface\hub\models--prajjwal1--bert-mini. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 0it [00:00, ?it/s]
DistilBertForSequenceClassification LOAD REPORT from: prajjwal1/bert-mini
Key      

In [21]:
previous_val_loss = float('inf')
for epoch in range(MAX_EPOCHS):
    # Entrainement
    model.train()
    for batch in tqdm(dataloader_train, desc=f"Epoch {epoch+1} - Training"):
        X_batch = batch['input_ids'].to(device)
        mask_batch = batch['attention_mask'].to(device)
        Y_batch = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            output = model(input_ids=X_batch, labels=Y_batch, attention_mask=mask_batch)
        scaler.scale(output.loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader_val, desc=f"Epoch {epoch+1} - Validation"):
            X_batch = batch['input_ids'].to(device)
            mask_batch = batch['attention_mask'].to(device)
            Y_batch = batch['labels'].to(device)
            output = model(input_ids=X_batch, labels=Y_batch, attention_mask=mask_batch)
            predicted_value = output.logits.argmax(dim=1)
            correct += (Y_batch == predicted_value).sum().item()
            total += Y_batch.size(0)
            val_loss += output.loss.item()
    val_accuracy = correct / total
    val_loss /= len(dataloader_val)

    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | val_loss: {val_loss:.4f} | val_accuracy: {val_accuracy:.4f} | lr: {optimizer.param_groups[0]['lr']:.6f}")
    if (previous_val_loss - val_loss) < 0.005:
        break
    previous_val_loss = val_loss

Epoch 1 - Training:   0%|          | 0/14212 [00:00<?, ?it/s]C:\Users\maxim\AppData\Local\Temp\ipykernel_20500\1416088295.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
Epoch 1 - Validation: 100%|██████████| 3553/3553 [00:59<00:00, 59.40it/s]


Epoch 1/4 | val_loss: 0.7269 | val_accuracy: 0.7264 | lr: 0.000015


Epoch 2 - Validation: 100%|██████████| 3553/3553 [01:00<00:00, 58.81it/s]


Epoch 2/4 | val_loss: 0.7057 | val_accuracy: 0.7357 | lr: 0.000010


Epoch 3 - Validation: 100%|██████████| 3553/3553 [00:59<00:00, 59.88it/s]

Epoch 3/4 | val_loss: 0.7009 | val_accuracy: 0.7397 | lr: 0.000005


In [22]:
torch.save(model.state_dict(), "../model.pt")